In [1]:
import h5py as h5
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
import os
import matplotlib.dates as dates
import datetime
import wave
import struct
import omp
import matplotlib.animation as manimation

In [2]:
#Sensibilité : -26dBFS pour 104 dB soit 3.17 Pa
FS = 2**23
S = FS*10**(-26/20.) / 3.17
po = 20e-6
Fs = 10000
NumMicPlot = 1
plt.close('all')
datadir = './'

In [3]:
print('Initialisation BeamForming')
#StdBF###########################################################################
NbMems = 32
Fe = 10e3
Po = 20e-6    

Initialisation BeamForming


In [4]:
# Physical parameters
c0 = 343
rho0 = 1.2

In [5]:
# Topographic parameters
Lx = 10
Ly = 8
Lz = 2.2

In [6]:
# Microphones coordinates
Rm = np.load('Xm_Cohrmds.npy').T
Ym = Rm[0,:]
Xm = Rm[1,:]
Zm = np.ones((1, NbMems))*Lz
Rm = np.vstack((Rm, Zm))

In [7]:
nx = 6
ny = 5
xs = np.linspace(-Lx/2+Lx/12, Lx/2-Lx/12, nx)    # centres des box en x
ys = np.linspace(-Ly/2+Ly/5, Ly/2-Ly/5, ny)
#nx, ny= len(xs), len(ys)
[Xs, Ys] = np.meshgrid(xs,ys)
Xs = Xs.flatten()
Ys = Ys.flatten()
Zs = np.zeros_like(Xs)
Rs = np.array([Xs, Ys, Zs])
Rms = np.linalg.norm(Rs[:,np.newaxis,:]-Rm[:,:,np.newaxis], axis = 0)  # distances entre micros (Rm) et centres (Rs)
ns = nx*ny
dfen = 10
nfft = int(Fe/dfen)
f = np.fft.rfftfreq(nfft,1/Fe)

In [8]:
G = np.outer(f,Rms).reshape(f.shape[0],Rms.shape[0], Rms.shape[1])/c0
G = np.exp(1j*2*np.pi*G)

In [11]:
np.shape(f)

(501,)

In [9]:
np.shape(G)   # 501 fréq, 32 micros, 30 positions des centres

(501, 32, 30)

In [ ]:
NumSec = 100
Max = 0
BFF = np.zeros(8)
LpdB = np.zeros(1000)
Signal = np.empty(1000*1000)

In [ ]:
Sigs = ( np.random.rand(int(Fe/dfen), NbMems)*2 - 1 )*32000 / S

In [ ]:
Spec = np.fft.rfft(Sigs, axis=0)   # 32 spectres

In [ ]:
np.shape(Spec)

In [ ]:
SpecG = Spec[:, :, None]*G

In [ ]:
np.shape(SpecG)

In [ ]:
BFSpec = np.sum(SpecG,1)/NbMems

In [ ]:
np.shape(BFSpec)

In [ ]:
BFSig = np.fft.irfft(BFSpec, axis = 0)

In [ ]:
np.shape(BFSig)

In [ ]:
BF = np.std(BFSig,axis = 0)

In [ ]:
np.shape(BF)

In [ ]:
OMPfo = np.zeros((nx,ny))

In [ ]:
ifo  = np.argmax(np.mean(np.abs(Spec), axis=1)) 

In [ ]:
Dico = G[ifo,:,:]

In [ ]:
result = omp.omp(Dico, Spec[ifo,:], maxit = 1)

In [ ]:
OMPfo = result.coef+1e-12

In [ ]:
BF[Ys==0] = 0

In [ ]:
OMPfo[Ys==0] = 0

In [ ]:
Box1 = np.logical_and(Xs<-1.67, Ys<-0.8)
Box2 = np.logical_and(Xs>-1.67, Xs<1.67, Ys<-0.8)
Box3 = np.logical_and(Xs>1.67, Ys<-0.8)
Box4 = np.logical_and(Xs<-1.67, Ys>0.8)
Box5 = np.logical_and(Xs>-1.67, Xs<1.67, Ys>0.8)
Box6 = np.logical_and(Xs>1.67, Ys>0.8)

In [ ]:
BFF[0] = np.sum(OMPfo[Box1])
BFF[1] = np.sum(OMPfo[Box2])
BFF[2] = np.sum(OMPfo[Box3])
BFF[5] = np.sum(OMPfo[Box4])
BFF[6] = np.sum(OMPfo[Box5])
BFF[7] = np.sum(OMPfo[Box6])

In [ ]:
np.shape(OMPfo)

In [ ]:
np.size(xs)

In [ ]:
np.shape(np.meshgrid(xs,ys))